# 1. Setup and Installation

In [ ]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 \
    bitsandbytes>=0.43.0 accelerate>=0.29.0 datasets>=2.19.0 \
    huggingface_hub>=0.22.0 wandb

# 2. Configuration and Drive Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

DRIVE_ROOT    = '/content/drive/MyDrive/arxiv-llm-project'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/model/checkpoints/llama-3b-arxiv-lora'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

HF_TOKEN      = userdata.get('HF_TOKEN')
HF_MODEL_ID   = 'Navyasri12355/llama-3.2-3b-arxiv-lora'
DATASET_ID    = 'Navyasri12355/arxiv-qa-dataset'
BASE_MODEL    = 'meta-llama/Llama-3.2-3B-Instruct'

print("Drive mounted ✓")
print(f"Checkpoints → {CHECKPOINT_DIR}")

# 3. Data Loading and Preprocessing

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET_ID, token=HF_TOKEN)
print(raw)
print("Sample:", raw['train'][0])

# 4. Tokenizer and Data Formatting

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

SYSTEM_PROMPT = (
    "You are a knowledgeable research assistant specializing in machine learning "
    "and AI papers from arXiv, specifically focusing on computer science subfields such as "
    "Machine Learning (cs.LG), Computer Vision (cs.CV), Computation and Language (cs.CL), "
    "and Artificial Intelligence (cs.AI). Answer questions accurately based on the provided context."
)

def format_example(example):
    """Convert instruction/input/output → LLaMA chat format."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"{example['instruction']}\n\n{example['input']}".strip()},
        {"role": "assistant", "content": example['output']},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = raw['train'].map(format_example, remove_columns=raw['train'].column_names)

# Train / validation split (95/5)
split    = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds  = split['test']

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print("\nFormatted example:\n", train_ds[0]['text'][:500])

# 5. Model Loading and Quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print("Model loaded ✓")
print(f"Params: {model.num_parameters():,}")

# 6. PEFT (LoRA) Setup

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1–2% of total params are trainable

# 7. Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,       # effective batch = 16
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=25,
    evaluation_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,                  # keep last 3 checkpoints only
    load_best_model_at_end=True,
    report_to='none',                    # change to 'wandb' if you want tracking
    run_name='llama-3b-arxiv-lora',
    max_steps=-1,                        # -1 = run all epochs
)

# 8. Model Training

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=1024,
    packing=False,
    args=training_args,
)

print("Starting training...")
trainer.train()

# Save best checkpoint to Drive
trainer.save_model(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Adapter saved to {CHECKPOINT_DIR} ✓")

# 9. Inference with Fine-Tuned Model

In [ ]:
from peft import PeftModel

# Reload in inference mode
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map='auto', token=HF_TOKEN
)
ft_model = PeftModel.from_pretrained(base, CHECKPOINT_DIR)
ft_model.eval()

def ask(question, context=""):
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": f"{question}\n\n{context}".strip()},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = ft_model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print(ask("What is LoRA and why is it useful for fine-tuning large language models?"))

# 10. Publish Model to Hugging Face Hub

In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)

# Merge adapter into base model and push (for easier inference later)
merged = ft_model.merge_and_unload()
merged.push_to_hub(HF_MODEL_ID, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(HF_MODEL_ID, token=HF_TOKEN, private=False)

print(f"✅ Model published: https://huggingface.co/{HF_MODEL_ID}")

# 11. Save Training Loss Log

In [ ]:
import json, os

log_history = trainer.state.log_history
loss_log = [e for e in log_history if 'loss' in e]

log_path = f'{DRIVE_ROOT}/eval/training_loss.json'
os.makedirs(os.path.dirname(log_path), exist_ok=True)
with open(log_path, 'w') as f:
    json.dump(loss_log, f, indent=2)

print(f"Loss log saved → {log_path}")
print(f"Final train loss: {loss_log[-1].get('loss', 'N/A')}")